<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/Llama3_2_3B_hellaswag_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3.2-3B 한국어 Hellaswag 데이터 테스트
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference :
1. https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
2. https://ai.meta.com/blog/llama-3-2-connect-2024-vision-edge-mobile-devices/
3. https://huggingface.co/google/gemma-2b-it

## [한국어 Hellaswag] 데이터 : https://huggingface.co/datasets/davidkim205/ko_hellaswag

In [ ]:
!nvidia-smi

# Llama 3.2-3B-Instruct 모델 불러오기

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Hugging Face Token 설정
os.environ['HF_TOKEN'] = "Input Your Token"

model_id = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Loading {model_id}...")

# 2. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 패딩 토큰 설정 (에러 방지용)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Model Loaded Successfully on {model.device}")

In [ ]:
def generate_response(system_message, user_message, tokenizer, model):
  #system_message = LLM에 대한 설정
  #user_message = 유저 프롬프트
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,   # 항상 가장 확률이 높은 값으로 예측
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
print(model.device)

# 시스템 프롬프트 설정

In [ ]:
system_prompt = '너는 주어진 label과 context를 기반으로 4개의 선택지 중에 가장 그럴듯한 ending을 선택하는 임무를 가진 챗봇이야. \
                ending 후보중에서 가장 그럴듯한 ending을 선택하고, 출력은 다른 말은 하지말고 "[0, 1, 2, 3]" 중에 하나의 값으로 출력해줘.'

In [ ]:
# example 1
label = '자동차에서 얼음 제거'
context_prompt = '그러자 남자는 차창을 덮고 있는 눈 위에 글을 쓰고, 겨울옷을 입은 여자는 미소를 짓는다. 그러면'
ending_list = [ ", 남자는 앞 유리에 왁스를 추가하고 자릅니다.",
                ", 한 사람이 스키 리프트에 탑승하고, 두 명의 남자가 겨울 옷을 입은 사람의 머리를 받치고 우리 소녀들이 썰매를 탈 때 눈이 내립니다.",
                ", 남자는 그물로 짠 크리스마스 코트를 입습니다.",
                ", 남자는 차에 쌓인 눈을 치우는 일을 계속한다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 2
label = '베이킹 쿠키'
context_prompt = '흰 유니폼을 입은 여성 셰프가 커다란 주방에 쌓여 있는 베이킹 팬을 보여준다. 프라이팬'
ending_list = [ "달걀 노른자와 베이킹 소다를 함유하십시오.", "그런 다음 흑설탕을 뿌립니다.", "카운터의 여과기에 놓입니다.", "페이스트리로 채워져 오븐에 들어갑니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 3
label = '치어리딩'
context_prompt = '치어리딩 팀이 포스터를 들고 시작하자 마스코트가 뒤에서 달려갑니다. 그들'
ending_list = [ "결국 개들을 들판으로 데려와 이리저리 밀어냅니다.",
               "필드에서 운동 선수의 루틴을 수행하기 시작합니다.",
               "그런 다음 루틴을 시작하고 나머지 소녀들이 스턴트를 위해 소녀들을 들어 올리는 동안 일부 소녀들은 스트리머와 함께 달립니다.",
               "어쿠스틱 곡을 연주하기 시작합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 4
label = '팔씨름'
context_prompt = '두 명의 보디빌더 여성이 테이블에 앉아 있습니다. 그들	'
ending_list = [ "다이빙 기술에 대해 이야기하고 있으며, 근육의 힘으로 서로를 매수하고 있습니다.",
               "운동용 자전거로 운동하고 있습니다.",
                "팔씨름을 하며 이기기 위해 경쟁하고 있습니다.",
                "평행 막대에 표시됩니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 5
label = '양치질'
context_prompt = '한 어린 소년이 싱크대를 향해 걸어갑니다. 소년'
ending_list = [ "떨어지는 것은 그의 바지를 바닥에서 똥을 싸고 있습니다.",
               "입을 헹구기 위해 물을 세운다.",
              "세면대 앞에 서서 칫솔에 치약을 묻힌 후 양치질을 합니다.",
              "냄비에 컵을 헹구고 그 위에 잔을 얹습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 6
label = '화장하기'
context_prompt = '카메라를 향해 키스를 날리는 여성이 콘택트렌즈를 끼운다. 그녀'
ending_list = [ "지금 크리스마스 트리 옆에 서서 말하고 있습니다.",
               "눈꺼풀에 아이섀도를 바르고 문지르기 시작한다.",
                "비닐봉지를 들고 코에 대고 오른쪽 눈을 잡고 눈을 보여준다.",
                "오른쪽 눈에 콘택트렌즈를 끼운 다음 눈 사이에 작은 렌즈를 끼우고 다른 렌즈를 따라 한 쌍으로 끼웁니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 1

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 7
label = '용접'
context_prompt = '한 소년이 집 뒷마당 뒷마당에서 땜납을 사용하여 분해된 자전거를 수리하고 있습니다. 소년'
ending_list = [ "렌치를 사용하여 칼을 사용하여 림을 눌러 한 쌍의 자전거 바퀴를 수리합니다.",
               "세제와 크래프트 다리미 병을 깨뜨립니다.",
                "현장에 들어가 스포츠 보호 마스크를 착용한 채 장갑을 끼고 뒷계단에 있는 파란색 기계를 켜고 땜납을 집어 듭니다.",
                "어린 소년이 자전거를 조립하고 분해된 부품에서 땜납을 추출하는 것을 지켜봅니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 8
label = '손 씻기'
context_prompt = '그런 다음 물을 틀고 손을 헹구고 손가락을 구부려 비누가 모두 꺼졌는지 확인합니다. 그 후'
ending_list = [ "완료되면 그녀는 흰 수건을 잡고 손을 말리고 입을 닦습니다.",
                ", 그녀는 웅크린 자세로 내려가 싱크대 바닥을 철저히 문질러 먼지가 없는지 확인합니다.",
                "그녀는 조금 밖으로 나가 손을 씻기 시작했다.",
                ", 그녀는 침대 옆 바닥에 앉아 면도 크림으로 몸 전체를 문지르기 시작하고 빗으로 머리를 빗습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 9
label = '수영'
context_prompt = '사람들은 수영장으로 뛰어들어 수영을 시작합니다. 한 사람이 끝에 도달하여 고글을 벗습니다. 그들은'
ending_list = [ "카메라를 향해 손을 흔들며 미소를 짓는다.",
                "마지막을 향해 앞뒤로 수영을 시작합니다.",
                "다이빙을 마치고 다시 물에 들어갑니다.",
                "수영을 계속하면 수영장에서 작은 풍선 튜브를 타고 점프합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

In [ ]:
# example 10
label = '다트 던지기'
context_prompt = '검은 조끼를 입은 남자가 방에 서 있습니다. 그'
ending_list = [ "벽에 있는 다트 판에 다트를 던집니다.",
               "걸레로 머리를 맞는다.",
                "부지깽이로 불을 붙입니다.",
                "말하는 물체를 들고 있습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_2_inference_result)

# 모델별 정확도에 기반한 성능 비교

|   | Llama-3.2-3B-Instruct | gpt-4o-mini | Llama-3.2-1B-Instruct | Llama-3.1-8B-Instruct | gemma-2-2b-it |
|---|---|---|---|---|---|
| 1 | x | o | x | x | o |
| 2 | x | o | x | x | x |
| 3 | o | o | o | o | x |
| 4 | o | o | x | o | o |
| 5 | o | o | o | o | o |
| 6 | o | x | x | x | x |
| 7 | x | o | x | o | o |
| 8 | x | o | x | o | x |
| 9 | o | o | x | o | o |
| 10 | x | o | x | o | o |

In [ ]:
# 1. Llama-3.2-3B-Instruct : 50% - 5/10
# 2. GPT-4o-mini : 90% - 9/10
# 3. Llama-3.2-1B-Instruct : 20% - 2/10
# 4. Llama-3.1-8B-Instruct : 70% - 7/10
# 5. gemma-2-2b-it : 60% - 6/10

# GPT-4o-mini 실험결과

In [ ]:
# example 1 : [3]
# example 2 : [3]
# example 3 : [2]
# example 4 : [2]
# example 5 : [2]
# example 6 : [3]
# example 7 : [2]
# example 8 : [0]
# example 9 : [0]
# example 10 : [0]

# Llama-3.2-1B-Instruct 테스트

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"

llama3_2_tokenizer = AutoTokenizer.from_pretrained(model_id)
llama3_2_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
def generate_response(system_message, user_message, tokenizer, model):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,   # 항상 가장 확률이 높은 값으로 예측
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
print(llama3_2_model.device)

# 시스템 프롬프트 설정

In [ ]:
system_prompt = '너는 주어진 label과 context를 기반으로 4개의 선택지 중에 가장 그럴듯한 ending을 선택하는 임무를 가진 챗봇이야. \
                ending 후보중에서 가장 그럴듯한 ending을 선택하고, 출력은 다른 말은 하지말고 "[0, 1, 2, 3]" 중에 하나의 값으로 출력해줘.'

In [ ]:
# example 1
label = '자동차에서 얼음 제거'
context_prompt = '그러자 남자는 차창을 덮고 있는 눈 위에 글을 쓰고, 겨울옷을 입은 여자는 미소를 짓는다. 그러면'
ending_list = [ ", 남자는 앞 유리에 왁스를 추가하고 자릅니다.",
                ", 한 사람이 스키 리프트에 탑승하고, 두 명의 남자가 겨울 옷을 입은 사람의 머리를 받치고 우리 소녀들이 썰매를 탈 때 눈이 내립니다.",
                ", 남자는 그물로 짠 크리스마스 코트를 입습니다.",
                ", 남자는 차에 쌓인 눈을 치우는 일을 계속한다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 2
label = '베이킹 쿠키'
context_prompt = '흰 유니폼을 입은 여성 셰프가 커다란 주방에 쌓여 있는 베이킹 팬을 보여준다. 프라이팬'
ending_list = [ "달걀 노른자와 베이킹 소다를 함유하십시오.", "그런 다음 흑설탕을 뿌립니다.", "카운터의 여과기에 놓입니다.", "페이스트리로 채워져 오븐에 들어갑니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 3
label = '치어리딩'
context_prompt = '치어리딩 팀이 포스터를 들고 시작하자 마스코트가 뒤에서 달려갑니다. 그들'
ending_list = [ "결국 개들을 들판으로 데려와 이리저리 밀어냅니다.",
               "필드에서 운동 선수의 루틴을 수행하기 시작합니다.",
               "그런 다음 루틴을 시작하고 나머지 소녀들이 스턴트를 위해 소녀들을 들어 올리는 동안 일부 소녀들은 스트리머와 함께 달립니다.",
               "어쿠스틱 곡을 연주하기 시작합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 4
label = '팔씨름'
context_prompt = '두 명의 보디빌더 여성이 테이블에 앉아 있습니다. 그들	'
ending_list = [ "다이빙 기술에 대해 이야기하고 있으며, 근육의 힘으로 서로를 매수하고 있습니다.",
               "운동용 자전거로 운동하고 있습니다.",
                "팔씨름을 하며 이기기 위해 경쟁하고 있습니다.",
                "평행 막대에 표시됩니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 5
label = '양치질'
context_prompt = '한 어린 소년이 싱크대를 향해 걸어갑니다. 소년'
ending_list = [ "떨어지는 것은 그의 바지를 바닥에서 똥을 싸고 있습니다.",
               "입을 헹구기 위해 물을 세운다.",
              "세면대 앞에 서서 칫솔에 치약을 묻힌 후 양치질을 합니다.",
              "냄비에 컵을 헹구고 그 위에 잔을 얹습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 6
label = '화장하기'
context_prompt = '카메라를 향해 키스를 날리는 여성이 콘택트렌즈를 끼운다. 그녀'
ending_list = [ "지금 크리스마스 트리 옆에 서서 말하고 있습니다.",
               "눈꺼풀에 아이섀도를 바르고 문지르기 시작한다.",
                "비닐봉지를 들고 코에 대고 오른쪽 눈을 잡고 눈을 보여준다.",
                "오른쪽 눈에 콘택트렌즈를 끼운 다음 눈 사이에 작은 렌즈를 끼우고 다른 렌즈를 따라 한 쌍으로 끼웁니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 1

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 7
label = '용접'
context_prompt = '한 소년이 집 뒷마당 뒷마당에서 땜납을 사용하여 분해된 자전거를 수리하고 있습니다. 소년'
ending_list = [ "렌치를 사용하여 칼을 사용하여 림을 눌러 한 쌍의 자전거 바퀴를 수리합니다.",
               "세제와 크래프트 다리미 병을 깨뜨립니다.",
                "현장에 들어가 스포츠 보호 마스크를 착용한 채 장갑을 끼고 뒷계단에 있는 파란색 기계를 켜고 땜납을 집어 듭니다.",
                "어린 소년이 자전거를 조립하고 분해된 부품에서 땜납을 추출하는 것을 지켜봅니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 8
label = '손 씻기'
context_prompt = '그런 다음 물을 틀고 손을 헹구고 손가락을 구부려 비누가 모두 꺼졌는지 확인합니다. 그 후'
ending_list = [ "완료되면 그녀는 흰 수건을 잡고 손을 말리고 입을 닦습니다.",
                ", 그녀는 웅크린 자세로 내려가 싱크대 바닥을 철저히 문질러 먼지가 없는지 확인합니다.",
                "그녀는 조금 밖으로 나가 손을 씻기 시작했다.",
                ", 그녀는 침대 옆 바닥에 앉아 면도 크림으로 몸 전체를 문지르기 시작하고 빗으로 머리를 빗습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 9
label = '수영'
context_prompt = '사람들은 수영장으로 뛰어들어 수영을 시작합니다. 한 사람이 끝에 도달하여 고글을 벗습니다. 그들은'
ending_list = [ "카메라를 향해 손을 흔들며 미소를 짓는다.",
                "마지막을 향해 앞뒤로 수영을 시작합니다.",
                "다이빙을 마치고 다시 물에 들어갑니다.",
                "수영을 계속하면 수영장에서 작은 풍선 튜브를 타고 점프합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 10
label = '다트 던지기'
context_prompt = '검은 조끼를 입은 남자가 방에 서 있습니다. 그'
ending_list = [ "벽에 있는 다트 판에 다트를 던집니다.",
               "걸레로 머리를 맞는다.",
                "부지깽이로 불을 붙입니다.",
                "말하는 물체를 들고 있습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

# Llama-3.1-8B-Instruct 테스트

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Llama-3.1-8B-Instruct"

llama3_2_tokenizer = AutoTokenizer.from_pretrained(model_id)
llama3_2_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
def generate_response(system_message, user_message, tokenizer, model):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,   # 항상 가장 확률이 높은 값으로 예측
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
print(llama3_2_model.device)

# 시스템 프롬프트 설정

In [ ]:
system_prompt = '너는 주어진 label과 context를 기반으로 4개의 선택지 중에 가장 그럴듯한 ending을 선택하는 임무를 가진 챗봇이야. \
                ending 후보중에서 가장 그럴듯한 ending을 선택하고, 출력은 다른 말은 하지말고 "[0, 1, 2, 3]" 중에 하나의 값으로 출력해줘.'

In [ ]:
# example 1
label = '자동차에서 얼음 제거'
context_prompt = '그러자 남자는 차창을 덮고 있는 눈 위에 글을 쓰고, 겨울옷을 입은 여자는 미소를 짓는다. 그러면'
ending_list = [ ", 남자는 앞 유리에 왁스를 추가하고 자릅니다.",
                ", 한 사람이 스키 리프트에 탑승하고, 두 명의 남자가 겨울 옷을 입은 사람의 머리를 받치고 우리 소녀들이 썰매를 탈 때 눈이 내립니다.",
                ", 남자는 그물로 짠 크리스마스 코트를 입습니다.",
                ", 남자는 차에 쌓인 눈을 치우는 일을 계속한다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 2
label = '베이킹 쿠키'
context_prompt = '흰 유니폼을 입은 여성 셰프가 커다란 주방에 쌓여 있는 베이킹 팬을 보여준다. 프라이팬'
ending_list = [ "달걀 노른자와 베이킹 소다를 함유하십시오.", "그런 다음 흑설탕을 뿌립니다.", "카운터의 여과기에 놓입니다.", "페이스트리로 채워져 오븐에 들어갑니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 3
label = '치어리딩'
context_prompt = '치어리딩 팀이 포스터를 들고 시작하자 마스코트가 뒤에서 달려갑니다. 그들'
ending_list = [ "결국 개들을 들판으로 데려와 이리저리 밀어냅니다.",
               "필드에서 운동 선수의 루틴을 수행하기 시작합니다.",
               "그런 다음 루틴을 시작하고 나머지 소녀들이 스턴트를 위해 소녀들을 들어 올리는 동안 일부 소녀들은 스트리머와 함께 달립니다.",
               "어쿠스틱 곡을 연주하기 시작합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 4
label = '팔씨름'
context_prompt = '두 명의 보디빌더 여성이 테이블에 앉아 있습니다. 그들	'
ending_list = [ "다이빙 기술에 대해 이야기하고 있으며, 근육의 힘으로 서로를 매수하고 있습니다.",
               "운동용 자전거로 운동하고 있습니다.",
                "팔씨름을 하며 이기기 위해 경쟁하고 있습니다.",
                "평행 막대에 표시됩니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 5
label = '양치질'
context_prompt = '한 어린 소년이 싱크대를 향해 걸어갑니다. 소년'
ending_list = [ "떨어지는 것은 그의 바지를 바닥에서 똥을 싸고 있습니다.",
               "입을 헹구기 위해 물을 세운다.",
              "세면대 앞에 서서 칫솔에 치약을 묻힌 후 양치질을 합니다.",
              "냄비에 컵을 헹구고 그 위에 잔을 얹습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 6
label = '화장하기'
context_prompt = '카메라를 향해 키스를 날리는 여성이 콘택트렌즈를 끼운다. 그녀'
ending_list = [ "지금 크리스마스 트리 옆에 서서 말하고 있습니다.",
               "눈꺼풀에 아이섀도를 바르고 문지르기 시작한다.",
                "비닐봉지를 들고 코에 대고 오른쪽 눈을 잡고 눈을 보여준다.",
                "오른쪽 눈에 콘택트렌즈를 끼운 다음 눈 사이에 작은 렌즈를 끼우고 다른 렌즈를 따라 한 쌍으로 끼웁니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 1

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 7
label = '용접'
context_prompt = '한 소년이 집 뒷마당 뒷마당에서 땜납을 사용하여 분해된 자전거를 수리하고 있습니다. 소년'
ending_list = [ "렌치를 사용하여 칼을 사용하여 림을 눌러 한 쌍의 자전거 바퀴를 수리합니다.",
               "세제와 크래프트 다리미 병을 깨뜨립니다.",
                "현장에 들어가 스포츠 보호 마스크를 착용한 채 장갑을 끼고 뒷계단에 있는 파란색 기계를 켜고 땜납을 집어 듭니다.",
                "어린 소년이 자전거를 조립하고 분해된 부품에서 땜납을 추출하는 것을 지켜봅니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 8
label = '손 씻기'
context_prompt = '그런 다음 물을 틀고 손을 헹구고 손가락을 구부려 비누가 모두 꺼졌는지 확인합니다. 그 후'
ending_list = [ "완료되면 그녀는 흰 수건을 잡고 손을 말리고 입을 닦습니다.",
                ", 그녀는 웅크린 자세로 내려가 싱크대 바닥을 철저히 문질러 먼지가 없는지 확인합니다.",
                "그녀는 조금 밖으로 나가 손을 씻기 시작했다.",
                ", 그녀는 침대 옆 바닥에 앉아 면도 크림으로 몸 전체를 문지르기 시작하고 빗으로 머리를 빗습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 9
label = '수영'
context_prompt = '사람들은 수영장으로 뛰어들어 수영을 시작합니다. 한 사람이 끝에 도달하여 고글을 벗습니다. 그들은'
ending_list = [ "카메라를 향해 손을 흔들며 미소를 짓는다.",
                "마지막을 향해 앞뒤로 수영을 시작합니다.",
                "다이빙을 마치고 다시 물에 들어갑니다.",
                "수영을 계속하면 수영장에서 작은 풍선 튜브를 타고 점프합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

In [ ]:
# example 10
label = '다트 던지기'
context_prompt = '검은 조끼를 입은 남자가 방에 서 있습니다. 그'
ending_list = [ "벽에 있는 다트 판에 다트를 던집니다.",
               "걸레로 머리를 맞는다.",
                "부지깽이로 불을 붙입니다.",
                "말하는 물체를 들고 있습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

llama3_2_inference_result = generate_response(system_message=system_prompt,
                              user_message=test_prompt,
                              tokenizer=llama3_2_tokenizer,
                              model=llama3_2_model)
print(llama3_2_inference_result)

# gemma-2-2b-it 테스트


In [ ]:
# pip install accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

In [ ]:
system_prompt = '너는 주어진 label과 context를 기반으로 4개의 선택지 중에 가장 그럴듯한 ending을 선택하는 임무를 가진 챗봇이야. \
                ending 후보중에서 가장 그럴듯한 ending을 선택하고, 출력은 다른 말은 하지말고 "[0, 1, 2, 3]" 중에 하나의 값으로 출력해줘.'

In [ ]:
# example 1
label = '자동차에서 얼음 제거'
context_prompt = '그러자 남자는 차창을 덮고 있는 눈 위에 글을 쓰고, 겨울옷을 입은 여자는 미소를 짓는다. 그러면'
ending_list = [ ", 남자는 앞 유리에 왁스를 추가하고 자릅니다.",
                ", 한 사람이 스키 리프트에 탑승하고, 두 명의 남자가 겨울 옷을 입은 사람의 머리를 받치고 우리 소녀들이 썰매를 탈 때 눈이 내립니다.",
                ", 남자는 그물로 짠 크리스마스 코트를 입습니다.",
                ", 남자는 차에 쌓인 눈을 치우는 일을 계속한다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 1
label = '자동차에서 얼음 제거'
context_prompt = '그러자 남자는 차창을 덮고 있는 눈 위에 글을 쓰고, 겨울옷을 입은 여자는 미소를 짓는다. 그러면'
ending_list = [ ", 남자는 앞 유리에 왁스를 추가하고 자릅니다.",
                ", 한 사람이 스키 리프트에 탑승하고, 두 명의 남자가 겨울 옷을 입은 사람의 머리를 받치고 우리 소녀들이 썰매를 탈 때 눈이 내립니다.",
                ", 남자는 그물로 짠 크리스마스 코트를 입습니다.",
                ", 남자는 차에 쌓인 눈을 치우는 일을 계속한다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 2
label = '베이킹 쿠키'
context_prompt = '흰 유니폼을 입은 여성 셰프가 커다란 주방에 쌓여 있는 베이킹 팬을 보여준다. 프라이팬'
ending_list = [ "달걀 노른자와 베이킹 소다를 함유하십시오.", "그런 다음 흑설탕을 뿌립니다.", "카운터의 여과기에 놓입니다.", "페이스트리로 채워져 오븐에 들어갑니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 3

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 3
label = '치어리딩'
context_prompt = '치어리딩 팀이 포스터를 들고 시작하자 마스코트가 뒤에서 달려갑니다. 그들'
ending_list = [ "결국 개들을 들판으로 데려와 이리저리 밀어냅니다.",
               "필드에서 운동 선수의 루틴을 수행하기 시작합니다.",
               "그런 다음 루틴을 시작하고 나머지 소녀들이 스턴트를 위해 소녀들을 들어 올리는 동안 일부 소녀들은 스트리머와 함께 달립니다.",
               "어쿠스틱 곡을 연주하기 시작합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 4
label = '팔씨름'
context_prompt = '두 명의 보디빌더 여성이 테이블에 앉아 있습니다. 그들	'
ending_list = [ "다이빙 기술에 대해 이야기하고 있으며, 근육의 힘으로 서로를 매수하고 있습니다.",
               "운동용 자전거로 운동하고 있습니다.",
                "팔씨름을 하며 이기기 위해 경쟁하고 있습니다.",
                "평행 막대에 표시됩니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 5
label = '양치질'
context_prompt = '한 어린 소년이 싱크대를 향해 걸어갑니다. 소년'
ending_list = [ "떨어지는 것은 그의 바지를 바닥에서 똥을 싸고 있습니다.",
               "입을 헹구기 위해 물을 세운다.",
              "세면대 앞에 서서 칫솔에 치약을 묻힌 후 양치질을 합니다.",
              "냄비에 컵을 헹구고 그 위에 잔을 얹습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]

test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 6
label = '화장하기'
context_prompt = '카메라를 향해 키스를 날리는 여성이 콘택트렌즈를 끼운다. 그녀'
ending_list = [ "지금 크리스마스 트리 옆에 서서 말하고 있습니다.",
               "눈꺼풀에 아이섀도를 바르고 문지르기 시작한다.",
                "비닐봉지를 들고 코에 대고 오른쪽 눈을 잡고 눈을 보여준다.",
                "오른쪽 눈에 콘택트렌즈를 끼운 다음 눈 사이에 작은 렌즈를 끼우고 다른 렌즈를 따라 한 쌍으로 끼웁니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 1

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 7
label = '용접'
context_prompt = '한 소년이 집 뒷마당 뒷마당에서 땜납을 사용하여 분해된 자전거를 수리하고 있습니다. 소년'
ending_list = [ "렌치를 사용하여 칼을 사용하여 림을 눌러 한 쌍의 자전거 바퀴를 수리합니다.",
               "세제와 크래프트 다리미 병을 깨뜨립니다.",
                "현장에 들어가 스포츠 보호 마스크를 착용한 채 장갑을 끼고 뒷계단에 있는 파란색 기계를 켜고 땜납을 집어 듭니다.",
                "어린 소년이 자전거를 조립하고 분해된 부품에서 땜납을 추출하는 것을 지켜봅니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 2

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 8
label = '손 씻기'
context_prompt = '그런 다음 물을 틀고 손을 헹구고 손가락을 구부려 비누가 모두 꺼졌는지 확인합니다. 그 후'
ending_list = [ "완료되면 그녀는 흰 수건을 잡고 손을 말리고 입을 닦습니다.",
                ", 그녀는 웅크린 자세로 내려가 싱크대 바닥을 철저히 문질러 먼지가 없는지 확인합니다.",
                "그녀는 조금 밖으로 나가 손을 씻기 시작했다.",
                ", 그녀는 침대 옆 바닥에 앉아 면도 크림으로 몸 전체를 문지르기 시작하고 빗으로 머리를 빗습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 9
label = '수영'
context_prompt = '사람들은 수영장으로 뛰어들어 수영을 시작합니다. 한 사람이 끝에 도달하여 고글을 벗습니다. 그들은'
ending_list = [ "카메라를 향해 손을 흔들며 미소를 짓는다.",
                "마지막을 향해 앞뒤로 수영을 시작합니다.",
                "다이빙을 마치고 다시 물에 들어갑니다.",
                "수영을 계속하면 수영장에서 작은 풍선 튜브를 타고 점프합니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))

In [ ]:
# example 10
label = '다트 던지기'
context_prompt = '검은 조끼를 입은 남자가 방에 서 있습니다. 그'
ending_list = [ "벽에 있는 다트 판에 다트를 던집니다.",
               "걸레로 머리를 맞는다.",
                "부지깽이로 불을 붙입니다.",
                "말하는 물체를 들고 있습니다." ]
ending_0 = ending_list[0]
ending_1 = ending_list[1]
ending_2 = ending_list[2]
ending_3 = ending_list[3]


test_prompt = f'{system_prompt} label:{label}, context:{context_prompt}, ending_0:{ending_0}, ending_1:{ending_1}, ending_2:{ending_2}, ending_3:{ending_3}'
print(test_prompt)
# 정답 : 0

input_ids = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))